# AlphaFold3 GPU Benchmark (T4, free Colab)

Runs AlphaFold3 on the **same 118-residue toy sequence, empty MSA, seed=1**
as `af3_cpu_colab.ipynb` and the existing CPU run in `results/sweep/`, so
the CPU/GPU/TPU comparison in `results/sweep/af3_comparison.md` is
apples-to-apples on input, only the backend changes.

**Before running:** go to `Runtime > Change runtime type > T4 GPU`, then
run all cells in order.


## 1. Confirm the runtime matches this notebook

In [ ]:
!nvidia-smi

## 2. System dependencies (Colab has root, unlike the Stanford login node -- AF3 needs a real C++ toolchain to build `libcifpp`/pybind11 at install time)

In [ ]:
!apt-get -qq update && apt-get -qq install -y --no-install-recommends \
    build-essential cmake ninja-build zlib1g-dev libeigen3-dev \
    libpcre2-dev libboost-all-dev libbz2-dev zstd

## 3. Install `uv` and clone AlphaFold3

In [ ]:
!curl -fsSL https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

!git clone --depth 1 https://github.com/google-deepmind/alphafold3.git /content/alphafold3

## 4. Build AlphaFold3 (`uv sync` compiles `libcifpp` -- this is the step that needs the toolchain from step 2 -- then `uv run build_data` processes the Chemical Component Dictionary into the pickle `run_alphafold.py` expects at startup. Both steps are in DeepMind's own docker/Dockerfile; skipping the second one produces a FileNotFoundError for chemical_component_sets.pickle a few seconds into Step 7, which is exactly what happened on this project's first attempt.)

In [ ]:
%cd /content/alphafold3
!uv sync
!uv run build_data

## 5. Download + decompress AF3 weights (~1GB compressed / ~1.15GB decompressed; public direct download, no approval form, subject to DeepMind's Weights Terms of Use)

In [ ]:
!mkdir -p /content/af3_weights
!curl -fsSL -o /content/af3_weights/af3.bin.zst \
    https://storage.googleapis.com/alphafold3/af3.bin.zst
!zstd -d /content/af3_weights/af3.bin.zst -o /content/af3_weights/af3.bin

## 6. Build the input JSON -- same 118-residue toy sequence used throughout this project, empty MSA, seed=1 (identical to `src/make_af3_input.py`, so this run is directly comparable to the CPU/TPU runs)

In [ ]:
import json

TOY_SEQUENCE_118 = (
    "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKV"
    "KALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWD"
)

payload = {
    "name": "af3_toy_test",
    "modelSeeds": [1],
    "sequences": [
        {
            "protein": {
                "id": "A",
                "sequence": TOY_SEQUENCE_118,
                "unpairedMsa": "",
                "pairedMsa": "",
                "templates": [],
            }
        }
    ],
    "dialect": "alphafold3",
    "version": 1,
}

with open("/content/alphafold3/af3_toy_input.json", "w") as f:
    json.dump(payload, f, indent=2)

print(json.dumps(payload, indent=2))

## 7. Run it -- tagged `gpu-t4` so it slots into the same comparison table as the CPU/TPU runs

**Runs via `uv run`, not a bare `python3` call:** `uv sync` in step 4 built AlphaFold3 into an isolated `.venv`, invisible to the system Python. Calling `python3 run_alphafold.py` directly fails with `ModuleNotFoundError: No module named 'alphafold3'` in well under a second -- that's not a real (fast!) run, it's an immediate crash. `uv run` executes inside the venv where the package actually lives.

**This cell now fails loudly (an `AssertionError`, cell stops with a red error) if the run didn't actually work -- either a non-zero exit code, or a suspiciously fast wall-clock (<30s). Do not proceed past a red error here.**

In [ ]:
import os
# T4 (and other compute-capability-7.x GPUs) need a different XLA flag than the AF3 Dockerfile default -- see docker/Dockerfile comments in the AF3 repo. Without this, run_alphafold.py raises a ValueError before doing any work.
os.environ["XLA_FLAGS"] = "--xla_disable_hlo_passes=custom-kernel-fusion-rewriter"

import json, subprocess, time

output_dir = "/content/af3_output"
cmd = [
    "uv", "run", "python3", "run_alphafold.py",
    "--json_path=/content/alphafold3/af3_toy_input.json",
    "--model_dir=/content/af3_weights",
    f"--output_dir={output_dir}",
    "--norun_data_pipeline",
    "--flash_attention_implementation=xla",
    "--jax_backend=gpu",
]

t0 = time.time()
proc = subprocess.run(cmd, cwd="/content/alphafold3", capture_output=True, text=True)
elapsed = time.time() - t0
print(proc.stdout[-3000:])
print(proc.stderr[-3000:])
print(f"Total wall-clock: {elapsed:.2f}s")

# Fail LOUDLY instead of silently -- subprocess.run() does not raise on a
# non-zero exit code, so without this check a crashed run_alphafold.py
# (e.g. ModuleNotFoundError if run_alphafold.py is ever called without
# `uv run`) would print a traceback buried in the output above and this
# notebook would carry on to Step 8 as if nothing happened, producing a
# result_af3_*.json full of nulls with no visible error. This exact
# failure mode happened during this project's first attempt at this
# notebook -- do not remove this check.
assert proc.returncode == 0, (
    f"\n\n{'='*70}\nrun_alphafold.py FAILED (exit code {proc.returncode}) "
    f"after {elapsed:.2f}s.\nThis is NOT a successful run -- do not "
    f"proceed to Step 8. See the stderr printed above for the real error."
    f"\n{'='*70}"
)
assert elapsed > 30, (
    f"\n\n{'='*70}\nrun_alphafold.py returned exit code 0 but took only "
    f"{elapsed:.2f}s -- too fast to be a real 5-sample run (the existing "
    f"CPU result in this project took ~392s). Treat this as a failure and "
    f"inspect the output above before trusting it.\n{'='*70}"
)
print("\nLooks like a real run -- proceed to Step 8.")

## 8. Write the result JSON (same schema as `results/result_cpu-colab.json` / `results/result_gpu-t4.json`, copy this into `results/` in the repo)

In [ ]:
import json, glob

# AlphaFold3 nests output under a job-name subdirectory and prefixes every
# file with the job name (e.g. af3_toy_test_summary_confidences.json, not
# summary_confidences.json) -- an exact-filename glob finds nothing and
# silently produces an all-null result even after a fully successful run.
# This is a wildcard match specifically to avoid that.
summary_path = glob.glob(f"{output_dir}/**/*summary_confidences.json", recursive=True)
summary_path = [p for p in summary_path if '/seed-' not in p]  # top-level (best-sample) summary, not a per-sample one
summary = json.load(open(summary_path[0])) if summary_path else {}
if not summary_path:
    print("WARNING: no summary_confidences.json found under", output_dir, "-- ptm/ranking_score will be null. Check the output_dir structure manually.")

result = {
    "run_tag": "gpu-t4",
    "backend": "gpu",
    "model": "alphafold3",
    "num_residues": 118,
    "num_samples": 5,
    "total_inference_seconds": elapsed,
    "seconds_per_sample": elapsed / 5,
    "best_ranking_score": summary.get("ranking_score"),
    "ptm": summary.get("ptm"),
    "fraction_disordered": summary.get("fraction_disordered"),
    "has_clash": summary.get("has_clash"),
}

result_path = "/content/result_af3_gpu-t4.json"
json.dump(result, open(result_path, "w"), indent=2)
print(json.dumps(result, indent=2))

## 9. Print it (copy into `results/` in the repo) and download the result + the raw AF3 output files

In [ ]:
!cat /content/result_af3_gpu-t4.json

In [ ]:
from google.colab import files
files.download('/content/result_af3_gpu-t4.json')

# Raw AF3 output -- copy these into structure/ and results/sweep/ the same
# way the existing af3_toy_test_* files were added, then update
# results/sweep/af3_comparison.md Section 3 with the real numbers.
import glob
for pattern in ["**/*.cif", "**/summary_confidences.json", "**/ranking_scores.csv"]:
    for path in glob.glob(f"{output_dir}/{pattern}", recursive=True):
        print("Found:", path)
        files.download(path)